# Risk Parameters of Truflation EV Fiat Market

In [26]:
import numpy as np
import pandas as pd
import plotly.express as px
pd.options.plotting.backend = "plotly"

import funding
import impact
import liquidations as liq
import pricedrift as drift
import pystable
from tqdm import tqdm

Set data-specific parameters:

In [31]:
file_name = "/Users/fredericoteixeira/Projects/overlay/data/btc_ev_v12_fiat.csv"
periodicity = 86400  # 1 day in seconds
cap = 10  # cap on pay off, set by governance

## Understanding the Data

Load and plot the data:

In [35]:
df = pd.read_csv(file_name).set_index("date")
df[["close", "twap"]].plot()

## Funding Rate - `k`

Compute `k`, the funding related risk metric.

It means that with this `k`, if OI exists only on one side (the worst case scenario), then in $1-\alpha$% of cases, the OI will get drawn down to zero just due to funding over the course of the next `n` days.

The result, when multipled by `1e18`, yields the amount that needs to be sent to the smart contract.

## Impact - $\lambda$ and $\delta$

## Liquidations - `maintenanceMarginFraction`

## Liquidations - `maintenanceMarginBurnRate`

## Price drift - `priceDriftUpperLimit`

In [14]:
mus = drift.generic_mu_max(
    df.loc[:,"btc_ev_v12_fiat_index"].to_numpy(), periodicity, long_twap=periodicity
)

df_mus = pd.DataFrame(
    mus,
    columns=["mu_max"],
    index=drift.ALPHAS
)
df_mus.columns.name = "mu_max"
df_mus.index.name = "alphas"

df_mus.plot()


        fit params: alpha: 0.863511763194551, beta: -0.06269061423309624,
        mu: 0.00045944276826278673, sigma: 0.0017438665768314502
        

        rescaled params (1/t = 1.1574074074074073e-05):
        alpha: 0.863511763194551, beta: -0.06269061423309624,
        mu: 5.317624632671142e-09, sigma: 3.3475354762985313e-09
        


Based on the 99.9% confidence, this value should be `2485221000000`.

The choice for a 99.9% confidence follows the whitepaper. It claims $\alpha$ (one minus confidence level) should be very small.

In [21]:
mu_max = df_mus.loc[0.01, :]
print(mu_max * 1e18)

mu_max
mu_max    2.485221e+12
Name: 0.01, dtype: float64


Furthermore, $\mu_{max}=2.4852 \times 10^{-6}$ means if price goes up/down `23.95%` in a duration equal to the longer TWAP (ie, 1 day), then the market doesn’t honor this spiked price. 

Remark: Equation (58) on page 10 of the whitepaper describes the formula for getting the percentage $e^{86400 \times \mu_{max}}$.

In [22]:
np.exp(periodicity * mu_max)

mu_max
mu_max    1.239519
Name: 0.01, dtype: float64

If we select alpha = 0.05 here too like we’ve done everywhere else, then that percentage is 3.31% which is very small and pretty likely to get hit. 

In [25]:
np.exp(periodicity * df_mus.loc[0.05,:])

mu_max
mu_max    1.033193
Name: 0.05, dtype: float64